## Apriori Algorithm

In [29]:
import pandas as pd
data = {
    'transactions': [1,2,3,4,5,6,7,8,9],
    'itemset': [
        'I1,I2,I5',
        'I2,I4',
        'I2,I3',
        'I1,I2,I4',
        'I1,I3',
        'I2,I3',
        'I1,I3',
        'I1,I2,I3,I5',
        'I1,I2,I3'
    ]
}

data_df = pd.DataFrame(data)

min_support = 2
# set of candidates of C1
C1 = {'I1':0,'I2':0,'I3':0,'I4':0,'I5':0}

for itemset in data_df['itemset']:
    for item in itemset.split(','):
        C1[item] += 1

# set of candidates of L1
L1 = set()
for item, count in C1.items():
    if count >= 2:
        L1.add(item)


C2 = {}
for itemset1 in L1:
    for itemset2 in L1:
        if itemset1 != itemset2   and int(itemset1[1:]) < int(itemset2[1:]):
            C2[f'{itemset1},{itemset2}'] = 0


L2 = set()
for itemset in C2.keys():
    C2[itemset] = data_df['itemset'].apply(lambda x: all(item in x.split(',') for item in itemset.split(','))).sum()
    if C2[itemset] >= min_support:
        L2.add(itemset)



C3 = {}
for itemset1 in L2:
    for itemset2 in L2:
        if itemset1 != itemset2 and int(itemset1.split(',')[0][1:]) < int(itemset2.split(',')[0][1:]):
            items1 = set(itemset1.split(','))
            items2 = set(itemset2.split(','))
            # we can only merge two itemsets if they have 2 items in common, which means the union of the two itemsets should have 3 items
            # Also if they are not discard in previous step, they must have 2 items in common, so we can just check if the union of the two itemsets has 3 items
            if len(items1.union(items2)) == 3:
                C3[f'{",".join(sorted(items1.union(items2)))}'] = 0

L3 = set()
for itemset in C3.keys():
    C3[itemset] = data_df['itemset'].apply(lambda x: all(item in x.split(',') for item in itemset.split(','))).sum()
    if C3[itemset] >= min_support:
        L3.add(itemset)


C4 = {}
for itemset1 in L3:
    for itemset2 in L3:
        if itemset1 != itemset2 and int(itemset1.split(',')[0][1:]) < int(itemset2.split(',')[0][1:]):
            items1 = set(itemset1.split(','))
            items2 = set(itemset2.split(','))
            # we can only merge two itemsets if they have 3 items in common, which means the union of the two itemsets should have 4 items
            # Also if they are not discard in previous step, they must have 3 items in common, so we can just check if the union of the two itemsets has 4 items
            if len(items1.union(items2)) == 4:
                C4[f'{",".join(sorted(items1.union(items2)))}'] = 0


# So the Frequent Itemsets are L1, L2, L3, L4
print('Frequent Itemsets:')
print('L1:', L1)
print('L2:', L2)
print('L3:', L3)
print('C2',C2)
print('C3',C3)
data_df.head()


Frequent Itemsets:
L1: {'I1', 'I5', 'I2', 'I3', 'I4'}
L2: {'I1,I3', 'I2,I4', 'I1,I2', 'I2,I3', 'I2,I5', 'I1,I5'}
L3: {'I1,I2,I5', 'I1,I2,I3'}
C2 {'I1,I5': np.int64(2), 'I1,I2': np.int64(4), 'I1,I3': np.int64(4), 'I1,I4': np.int64(1), 'I2,I5': np.int64(2), 'I2,I3': np.int64(4), 'I2,I4': np.int64(2), 'I3,I5': np.int64(1), 'I3,I4': np.int64(0), 'I4,I5': np.int64(0)}
C3 {'I1,I2,I3': np.int64(2), 'I1,I2,I4': np.int64(1), 'I1,I2,I5': np.int64(2)}


,transactions,itemset
0,1,"I1,I2,I5"
1,2,"I2,I4"
2,3,"I2,I3"
3,4,"I1,I2,I4"
4,5,"I1,I3"


### Association Rule Mining
- Calculate support, confidence and lift 
- Total transactions \(N = 9\)  
- $Support(X)=\frac{count(X)}{9}$, $Confidence(X \rightarrow Y)=\frac{count(X \cup Y)}{count(X)}$, $Lift(X \rightarrow Y)=\frac{Confidence(X \rightarrow Y)}{Support(Y)}$

#### Rules from **L2** (2-itemsets)
##### From itemset {I1, I2}
##### From itemset {I1, I3}
##### From itemset {I2, I3}
##### From itemset {I2, I5}
#### Rules from **L3** (3-itemsets)
##### From itemset {I1, I2, I5}
##### From itemset {I1, I2, I3}

In [36]:
from itertools import combinations



def generate_rules(frequent_itemsets: list[str]):
    all_rules = []
    for itemset_str in frequent_itemsets:
        items = set(itemset_str.split(','))
        n = len(items)
        
        # A rule A -> B must have A as a subset of items
        # The size of A can range from 1 to (n-1)
        for i in range(1, n):
            for antecedent in combinations(items, i):
                antecedent = set(antecedent)
                consequent = items - antecedent
                
                rule = f"{sorted(list(antecedent))} -> {sorted(list(consequent))}"
                all_rules.append(rule)
                
    return all_rules

# Generate for L2 and L3
rules_l2 = generate_rules(list(L2))
rules_l3 = generate_rules(list(L3))
all_rule_combinations = rules_l2 + rules_l3

num_transactions = 9
all_counts = {**C1, **C2, **C3}

def get_count(item_set):
    """Helper to fetch count from dictionaries regardless of item order"""
    key = ",".join(sorted(list(item_set)))
    
    return all_counts.get(key, 0)




for rule_str in all_rule_combinations:
    # Logic to extract sets from the string format "['I1'] -> ['I2']"
    parts = rule_str.split(" -> ")
    # Using strip and split to turn "['I1', 'I2']" into {'I1', 'I2'}
    ant = set(parts[0].strip("[]").replace("'", "").replace(" ", "").split(","))
    cons = set(parts[1].strip("[]").replace("'", "").replace(" ", "").split(","))
    both = ant.union(cons)
    # Fetch Counts
    count_ant = get_count(ant)
    count_cons = get_count(cons)
    count_both = get_count(both)
    
    if count_ant > 0:
        # 1. Support = count(both) / N
        support_both = count_both / num_transactions
        
        # 2. Confidence = count(both) / count(ant)
        confidence = count_both / count_ant
        
        # 3. Lift = Confidence / Support(cons)
        support_cons = count_cons / num_transactions
        lift = confidence / support_cons
        
        # Output only if confidence is meaningful (e.g., > 0)
        if confidence > 0:
            print(f"{rule_str:<25} | {confidence:.2f} | {lift:.2f}")


['I1'] -> ['I3']          | 0.67 | 1.00
['I3'] -> ['I1']          | 0.67 | 1.00
['I4'] -> ['I2']          | 1.00 | 1.29
['I2'] -> ['I4']          | 0.29 | 1.29
['I1'] -> ['I2']          | 0.67 | 0.86
['I2'] -> ['I1']          | 0.57 | 0.86
['I3'] -> ['I2']          | 0.67 | 0.86
['I2'] -> ['I3']          | 0.57 | 0.86
['I5'] -> ['I2']          | 1.00 | 1.29
['I2'] -> ['I5']          | 0.29 | 1.29
['I1'] -> ['I5']          | 0.33 | 1.50
['I5'] -> ['I1']          | 1.00 | 1.50
['I1'] -> ['I2', 'I5']    | 0.33 | 1.50
['I5'] -> ['I1', 'I2']    | 1.00 | 2.25
['I2'] -> ['I1', 'I5']    | 0.29 | 1.29
['I1', 'I5'] -> ['I2']    | 1.00 | 1.29
['I1', 'I2'] -> ['I5']    | 0.50 | 2.25
['I2', 'I5'] -> ['I1']    | 1.00 | 1.50
['I1'] -> ['I2', 'I3']    | 0.33 | 0.75
['I3'] -> ['I1', 'I2']    | 0.33 | 0.75
['I2'] -> ['I1', 'I3']    | 0.29 | 0.64
['I1', 'I3'] -> ['I2']    | 0.50 | 0.64
['I1', 'I2'] -> ['I3']    | 0.50 | 0.75
['I2', 'I3'] -> ['I1']    | 0.50 | 0.75
